# Cheat Sheet — Bayes' Theorem, LDA & QDA

Quick reference for formulas, assumptions, sklearn API, and decision rules. Runnable snippets are
included but this notebook is meant to be *skimmed*, not executed top to bottom like a lab.


## 1. Bayes' Theorem

$$P(B\mid A) = \frac{P(A\mid B)\,P(B)}{P(A)}$$

| Term | Name | Meaning |
|---|---|---|
| $P(B)$ | prior | belief before evidence |
| $P(B\mid A)$ | posterior | belief after evidence $A$ |
| $P(A\mid B)$ | likelihood | probability of evidence given $B$ is true |
| $P(A)$ | evidence / normalizer | total probability of the evidence across all cases |

**Total probability trick** (when $B$ has only two outcomes, e.g. sick / not sick):
$$P(A) = P(A\mid B)P(B) + P(A\mid \lnot B)P(\lnot B)$$


In [1]:
def bayes_update(prior, true_positive_rate, false_positive_rate):
    p_not = 1 - prior
    p_evidence = true_positive_rate * prior + false_positive_rate * p_not
    return (true_positive_rate * prior) / p_evidence

# usage
bayes_update(prior=0.2, true_positive_rate=0.95, false_positive_rate=0.3)


0.4418604651162791

## 2. LDA vs. QDA — side by side

| | **LDA** | **QDA** |
|---|---|---|
| Covariance assumption | **one shared** covariance matrix $\Sigma$ across all classes | **each class gets its own** $\Sigma_k$ |
| Decision boundary shape | linear (straight line / hyperplane) | quadratic (curved — ellipse, parabola, hyperbola) |
| Parameters to estimate | $O(p^2)$ total (one shared matrix) | $O(K\cdot p^2)$ total (one matrix per class) |
| Data efficiency | more data-efficient, lower variance | needs more data per class, higher variance |
| Bias | higher (wrong boundary shape if classes truly differ) | lower (flexible shape) |
| Good default when | classes have similar spread/shape | classes have visibly different spread/shape and you have enough data per class |
| Supervised dim. reduction | yes — projects to up to $K-1$ axes | no (not typically used this way) |
| sklearn class | `LinearDiscriminantAnalysis` | `QuadraticDiscriminantAnalysis` |

### Shared assumptions (both LDA and QDA)
1. Each class's features are drawn from a **multivariate Gaussian**.
2. Observations are **independent** (watch out for repeated-measures / time-series / clustered data).
3. (LDA only) **Equal covariance** across classes.


## 3. Discriminant functions

**LDA (linear in $x$):**
$$\delta_k(x) = x^T \Sigma^{-1}\mu_k - \tfrac12 \mu_k^T \Sigma^{-1}\mu_k + \log \pi_k$$

**QDA (quadratic in $x$, note the class-specific $\Sigma_k$):**
$$\log p(y=k\mid x) = \log \pi_k - \tfrac12\log|2\pi\Sigma_k| - \tfrac12 (x-\mu_k)^T\Sigma_k^{-1}(x-\mu_k) + C$$

Predicted class = $\arg\max_k \delta_k(x)$ (LDA) or $\arg\max_k \log p(y=k\mid x)$ (QDA).


## 4. sklearn API quick reference

```python
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis

lda = LinearDiscriminantAnalysis(
    solver="svd",          # 'svd' (default, no shrinkage), 'lsqr', or 'eigen' (both support shrinkage)
    shrinkage=None,        # None, 'auto', or a float in [0,1] — use with 'lsqr'/'eigen' for high-dim/small-n data
    priors=None,           # override class priors, e.g. [0.5, 0.5] to ignore observed imbalance
    n_components=None,     # for dimensionality reduction: number of discriminant axes to keep (<= K-1)
)

qda = QuadraticDiscriminantAnalysis(
    priors=None,
    reg_param=0.0,         # shrinkage toward a shared covariance — raise this if covariance is singular/unstable
    store_covariance=True, # keep per-class covariance matrices on .covariance_
)

lda.fit(X_train, y_train)
lda.predict(X_test)             # hard class labels
lda.predict_proba(X_test)       # posterior probabilities per class
lda.decision_function(X_test)   # raw discriminant scores
lda.transform(X_test)           # project onto discriminant axes (LDA only, needs n_components or K>2)
lda.fit_transform(X_train, y_train)  # fit AND reduce dimensions in one call
```


## 5. Preprocessing checklist

- [ ] **Scale continuous features** (`StandardScaler`) — LDA/QDA are sensitive to feature scale
      because they rely on covariance, which is scale-dependent.
- [ ] **Fit the scaler on train only**, then `.transform()` (never `.fit_transform()`) the test set.
- [ ] **One-hot encode categoricals** — but remember this technically breaks the Gaussian
      assumption for those columns; it's a common, usually-tolerable approximation, not a purist's
      choice.
- [ ] **Check class balance.** Neither model handles severe imbalance well; consider
      `priors=[...]`, resampling, or a different metric (AUC, F1) instead of accuracy.
- [ ] **Check for near-duplicate/collinear features** — collinearity can make covariance matrices
      singular or unstable, especially for QDA (more parameters, less data per estimate).


## 6. Assumption-checking snippets

In [2]:
# Normality per class (visual + numeric)
import scipy.stats as stats
import seaborn as sns
import matplotlib.pyplot as plt

def check_normality(df, feature, target, classes=(0, 1)):
    fig, ax = plt.subplots(figsize=(6, 4))
    for c in classes:
        vals = df.loc[df[target] == c, feature]
        stat, p = stats.shapiro(vals.sample(min(len(vals), 5000), random_state=0))
        sns.kdeplot(vals, ax=ax, label=f"class {c} (Shapiro p={p:.3g})")
    ax.set_title(f"{feature} distribution by class")
    ax.legend()
    plt.show()

# usage: check_normality(df, "some_feature", "target")


In [3]:
# Equal-covariance check (relative Frobenius-norm difference)
import numpy as np

def covariance_similarity(df, features, target, classes=(0, 1)):
    covs = [df.loc[df[target] == c, features].cov().values for c in classes]
    diff = np.linalg.norm(covs[0] - covs[1], ord='fro')
    avg = np.mean([np.linalg.norm(c, ord='fro') for c in covs])
    return diff / avg  # rule of thumb: well under ~0.3-0.4 supports the LDA assumption

# usage: covariance_similarity(df, ["f1", "f2", "f3"], "target")


## 7. Evaluation snippets

In [4]:
from sklearn.metrics import (confusion_matrix, ConfusionMatrixDisplay, precision_score,
                              recall_score, accuracy_score, roc_auc_score, roc_curve,
                              classification_report)

def evaluate_binary(model, X, y_true, label=""):
    y_pred = model.predict(X)
    print(f"--- {label} ---")
    print("Precision:", round(precision_score(y_true, y_pred), 4))
    print("Recall:   ", round(recall_score(y_true, y_pred), 4))
    print("Accuracy: ", round(accuracy_score(y_true, y_pred), 4))
    ConfusionMatrixDisplay(confusion_matrix(y_true, y_pred)).plot()
    plt.title(label)
    plt.show()

def plot_roc(model, X_test, y_test):
    proba = model.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, proba)
    auc = roc_auc_score(y_test, proba)
    plt.plot(fpr, tpr, label=f"AUC={auc:.3f}")
    plt.plot([0, 1], [0, 1], "--", color="grey")
    plt.xlabel("FPR"); plt.ylabel("TPR"); plt.legend()
    plt.show()


In [5]:
from sklearn.model_selection import cross_val_score, StratifiedKFold

def cv_score(model, X, y, scoring="accuracy", folds=5):
    cv = StratifiedKFold(n_splits=folds, shuffle=True, random_state=42)
    scores = cross_val_score(model, X, y, cv=cv, scoring=scoring)
    print(f"{scoring}: {scores.mean():.3f} +/- {scores.std():.3f}")
    return scores


## 8. Decision boundary plotting (2D only)

In [6]:
import numpy as np
import matplotlib.pyplot as plt

def plot_2d_boundary(model, X2, y2, colors=('red', 'green', 'blue'), title=""):
    """X2 must be shape (n, 2). Works for any classifier with .predict()."""
    x_min, x_max = X2[:, 0].min() - 0.5, X2[:, 0].max() + 0.5
    y_min, y_max = X2[:, 1].min() - 0.5, X2[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 400), np.linspace(y_min, y_max, 400))
    Z = model.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

    fig, ax = plt.subplots(figsize=(7, 7))
    ax.contourf(xx, yy, Z, alpha=0.15, colors=colors)
    ax.contour(xx, yy, Z, colors=colors, linewidths=1)
    for i, c in enumerate(sorted(set(y2))):
        pts = X2[y2 == c]
        ax.scatter(pts[:, 0], pts[:, 1], c=colors[i], s=30, label=str(c))
    ax.legend()
    ax.set_title(title)
    plt.show()


## 9. Decision guide — which model should I reach for?

```
Is the decision boundary between classes visibly curved / classes have
very different spread or orientation in feature space?
│
├── Yes → try QDA (need enough samples per class: rule of thumb, at least
│         several times the number of features, per class)
│
└── No / unsure → start with LDA (fewer parameters, more stable) or
          Logistic Regression (no distributional assumption on X at all,
          often a stronger baseline when assumptions are shaky)

Is a class rare, or the feature distributions clearly non-Gaussian
(counts, heavy skew, multimodal)?
│
├── Yes → prefer Logistic Regression, tree ensembles, or resampling +
│         careful metric choice (AUC/F1, not raw accuracy)
│
└── No → LDA/QDA remain reasonable, fast, interpretable options
```

**Naive Bayes note:** Gaussian Naive Bayes is QDA with the *off-diagonal* covariance terms forced
to zero (features assumed independent within a class). It's a further bias/variance step past QDA
— even more data-efficient, even more biased if features are correlated.


## 10. Common pitfalls

- Fitting a **new scaler on the test set** instead of reusing the training scaler (data leakage).
- Feeding **un-scaled** mixed-magnitude features into LDA/QDA and being confused by poor results.
- Using **QDA on a class with very few samples** — the covariance estimate becomes singular/unstable; sklearn may warn or silently produce poor results. Consider `reg_param` or fall back to LDA.
- Treating a small **Shapiro-Wilk p-value** as an automatic disqualifier — at large $n$ it will
  almost always reject perfect normality. Look at the actual shape (KDE/histogram/QQ-plot), not just the p-value.
- Reporting only **accuracy** on an imbalanced target — use precision/recall/AUC as well.
- Forgetting that **one-hot encoded columns are not Gaussian** — a known, generally-accepted approximation, not a violation to "fix" by trying to transform them.
